In [1]:
# =============================================================================
# REAL-TIME VOICE AGENT — earpiece Correction-GPT (live use case)
# =============================================================================
#
# ---------------------------------------------------------------------------
# PRODUCT PICTURE (why this notebook exists)
# ---------------------------------------------------------------------------
# On a live sales call, you wear an earpiece copilot:
#
#   mic  →  hear what was SAID
#   brain → Correction-GPT / tools (is it wrong vs ground truth?)
#   ear   → quiet spoken fix ("Correction: price is $299…")
#
# End-to-end pipeline we are wiring:
#
#   ┌──────────┐   ┌─────────┐   ┌──────────┐   ┌────────────┐   ┌─────┐
#   │ Mic PCM  │ → │   VAD   │ → │ Whisper  │ → │ Correction │ → │ TTS │
#   │ chunks   │   │ speech? │   │ speech→  │   │ GPT/tools  │   │ ear │
#   └──────────┘   └─────────┘   │ text     │   └────────────┘   └─────┘
#                                └──────────┘
#
# Latency goal called out in the course: keep the chain snappy
# (target < ~500ms feels "live"; real systems tune model size, VAD, streaming).
#
# ---------------------------------------------------------------------------
# ABBREVIATIONS (full form + one sentence)
# ---------------------------------------------------------------------------
#   VAD  = Voice Activity Detection
#          Decides if the mic is hearing speech or just silence/noise.
#
#   ASR  = Automatic Speech Recognition
#          Turns spoken audio into text (Whisper does this here).
#
#   TTS  = Text-To-Speech
#          Turns correction text back into sound for the earpiece.
#
#   PCM  = Pulse Code Modulation
#          The raw digital audio samples coming off the microphone.
#
#   MCP  = Model Context Protocol  (from earlier notebook)
#          Standard way for the agent to call restricted tools (price lookup, etc.).
#
#   SFT  = Supervised Fine-Tuning  (from earlier notebook)
#          Training on flashcards so the model learns to give corrections.
#
# ---------------------------------------------------------------------------
# IMPORTS — what each library is for
# ---------------------------------------------------------------------------
#   pyaudio     — open the microphone, read PCM chunks
#   webrtcvad   — Google WebRTC VAD (speech vs non-speech on short frames)
#   wave        — optional save/load .wav for debugging
#   whisper     — ASR model (speech → text)
#   pyttsx3     — offline TTS (text → speaker)
#   numpy       — array math on audio samples
#   torch       — if Correction-GPT / MiniGPT runs on GPU/CPU tensors
#   time        — timestamps for "silence lasted 0.8s → utterance done"
#   deque       — ring buffer of recent chunks (optional streaming tricks)
#   threading   — run capture / ASR / TTS without freezing each other
#   queue       — thread-safe handoff: audio→transcript→correction→speech
#

import pyaudio
import webrtcvad
import wave
import whisper
import pyttsx3
import numpy as np
import torch
import time
from collections import deque
import threading
import queue

print("Voice-agent deps imported.")
print("Next: VAD wrapper — decide WHEN an utterance is complete.")


/Users/girish11/aifromscratch_code/.venv/lib/python3.11/site-packages/webrtcvad.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Voice-agent deps imported.
Next: VAD wrapper — decide WHEN an utterance is complete.


In [2]:
# =============================================================================
# VAD WRAPPER — detect speech start/stop, return one full utterance
# =============================================================================
#
# ---------------------------------------------------------------------------
# WHY VAD? (easy story)
# ---------------------------------------------------------------------------
# Mic is always on. Most of the time: silence, keyboard, AC hum.
# You only want Whisper when the user FINISHED a phrase.
#
# State machine picture:
#
#   IDLE  --speech frame-->  SPEAKING (collect chunks in speech_buffer)
#     ^                           |
#     |                           |  silence longer than silence_duration
#     +------- return full_audio -+
#
#   process() returns:
#     None         → still listening / mid-sentence
#     bytes blob   → "utterance ready" → send to Whisper
#
# ---------------------------------------------------------------------------
# WEBRTC VAD NOTES
# ---------------------------------------------------------------------------
#   aggressiveness 0–3: higher = more aggressive at labeling noise as non-speech
#     0 = sensitive (more speech) … 3 = strict (more silence)
#   Frames must be 10, 20, or 30 ms of 16-bit mono PCM at 8/16/32/48 kHz.
#   We standardize on 16 kHz (Whisper-friendly).
#
# Note: silence_threshold is reserved for an optional energy gate; this wrapper
# leans on WebRTC's is_speech for the binary decision.
#

class VAD:
    def __init__(self, aggressiveness=2):
        # WebRTC engine: classifies each short frame as speech / not-speech
        self.vad = webrtcvad.Vad(aggressiveness)
        self.silence_threshold = 500   # optional energy hint (not used in is_speech path)
        self.silence_duration = 0.8    # seconds of quiet after speech → end of utterance
        self.speech_buffer = []        # list of raw PCM byte frames while talking
        self.is_speaking = False       # state flag: inside an utterance?
        self.last_speech_time = 0      # timestamp of most recent speech frame
        self.sample_rate = 16000       # Hz — must match mic + Whisper setup

    def is_speech(self, audio_chunk):
        """True if this tiny PCM frame sounds like speech."""
        # audio_chunk: bytes, 16 kHz mono PCM (correct duration for webrtcvad)
        return self.vad.is_speech(audio_chunk, self.sample_rate)

    def process(self, audio_bytes, timestamp):
        """Feed one frame. Return full utterance bytes when speech ends, else None."""
        if self.is_speech(audio_bytes):
            # Rising edge: speech just started → clear buffer and collect
            if not self.is_speaking:
                self.is_speaking = True
                self.speech_buffer = []
            self.speech_buffer.append(audio_bytes)
            self.last_speech_time = timestamp
        else:
            # Quiet frame: if we were speaking and quiet lasted long enough → done
            if self.is_speaking and (timestamp - self.last_speech_time > self.silence_duration):
                self.is_speaking = False
                full_audio = b"".join(self.speech_buffer)
                self.speech_buffer = []
                return full_audio  # complete utterance for Whisper
        return None  # keep listening


# Tiny mental demo (no mic yet):
#   frames:  . . S S S S S . . . .     (S=speech, .=silence)
#   state:   idle  speaking……  → after 0.8s quiet → emit joined S frames
print("VAD class ready | sample_rate=16k | silence_duration=0.8s | aggressiveness default=2")


VAD class ready | sample_rate=16k | silence_duration=0.8s | aggressiveness default=2


In [3]:
# =============================================================================
# AUDIO STREAMING — read the mic in tiny chunks forever
# =============================================================================
# Picture:
#   Mic ──► PyAudio ──► 30ms PCM chunks ──► queue ──► VAD thread/loop
#
# Why a queue?
#   This function blocks on stream.read(). Put it on its own thread so the
#   main loop can still run VAD / Whisper / TTS.
#
# Settings:
#   sample_rate=16000  → matches VAD + Whisper
#   chunk_duration=0.03 → 30 ms frames (WebRTC VAD-friendly)
#   paInt16 + channels=1 → 16-bit mono PCM
#

def audio_stream(queue, sample_rate=16000, chunk_duration=0.03):
    """Continuously capture mic audio and push PCM bytes into queue."""
    p = pyaudio.PyAudio()
    frames_per_buffer = int(sample_rate * chunk_duration)
    stream = p.open(
        format=pyaudio.paInt16,
        channels=1,
        rate=sample_rate,
        input=True,
        frames_per_buffer=frames_per_buffer,
    )
    print("Listening...")
    try:
        while True:
            data = stream.read(frames_per_buffer, exception_on_overflow=False)
            queue.put(data)  # hand off to main loop
    except KeyboardInterrupt:
        pass
    finally:
        stream.stop_stream()
        stream.close()
        p.terminate()


print("audio_stream() ready — run it inside a background thread from main().")


audio_stream() ready — run it inside a background thread from main().


In [4]:
# =============================================================================
# WHISPER ASR — speech bytes → text (utterance-level, not true streaming)
# =============================================================================
# Sticky:
#   Whisper wants a whole clip (or file), not one 30ms frame.
#   So VAD first gathers a full utterance → then we transcribe once.
#
#   model_size='tiny.en' → smallest English model (faster, good for demos).
#   Production low-latency ASR often uses streaming APIs (Deepgram, etc.).
#

class ASR:
    def __init__(self, model_size="tiny.en"):
        # Downloads weights the first time you run this
        self.model = whisper.load_model(model_size)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

    def transcribe(self, audio_bytes):
        """PCM utterance bytes → transcript string."""
        # Whisper's file API is simplest: write a temp 16 kHz mono WAV
        with wave.open("temp.wav", "wb") as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)      # 16-bit = 2 bytes per sample
            wf.setframerate(16000)
            wf.writeframes(audio_bytes)
        result = self.model.transcribe("temp.wav", fp16=(self.device == "cuda"))
        return result["text"].strip()


print("ASR class ready | default model=tiny.en")


ASR class ready | default model=tiny.en


In [5]:
# =============================================================================
# TTS — correction text → spoken audio in the ear
# =============================================================================
# pyttsx3 = offline system TTS (fast, no cloud key).
# speak() starts a daemon thread so the main loop keeps listening.
#

class TTS:
    def __init__(self):
        self.engine = pyttsx3.init()
        self.engine.setProperty("rate", 180)   # words per minute
        self.engine.setProperty("volume", 0.8)

    def speak(self, text):
        """Say text without blocking the mic loop."""
        def _speak():
            self.engine.say(text)
            self.engine.runAndWait()

        threading.Thread(target=_speak, daemon=True).start()


print("TTS class ready | offline pyttsx3")


TTS class ready | offline pyttsx3


In [6]:
# =============================================================================
# CORRECTION ENGINE — transcript + ground truth → short fix line
# =============================================================================
# Tries week2/mini_gpt.py + week2/tokenizer.py (NOT day8_minigpt).
# If DPO/SFT weights are missing or vocab doesn't match, uses a simple
# rule-based fallback so the live loop still works for learning.
#

from pathlib import Path
import sys

_WEEK2 = Path.cwd() / "week2"
if _WEEK2.is_dir():
    sys.path.insert(0, str(_WEEK2.resolve()))
elif Path.cwd().name == "week2":
    sys.path.insert(0, str(Path.cwd().resolve()))
else:
    sys.path.insert(0, str(Path("week2").resolve()))

from tokenizer import BPETokenizer
from mini_gpt import MiniGPT
import torch

USE_MODEL = False
tokenizer = None
model = None


def format_prompt(ground_truth, user_text):
    """Minimal instruction prompt if a trained model IS loaded."""
    return (
        "You are an AI sales coach. Whisper a quiet correction.\n\n"
        f"Ground truth: {ground_truth}\n"
        f"User said: {user_text}\n\n"
        "### Response:\n"
    )


def generate_correction(ground_truth, user_text):
    """Return a short coach line for the earpiece."""
    if USE_MODEL and model is not None and tokenizer is not None:
        prompt = format_prompt(ground_truth, user_text)
        input_ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long)
        with torch.no_grad():
            out_ids = model.generate(input_ids, max_new_tokens=20, temperature=0.7)
        text = tokenizer.decode(out_ids[0].tolist())
        if "### Response:" in text:
            return text.split("### Response:")[-1].strip()
        return text.strip()

    # Fallback: naive "words in ground truth but not in user text"
    truth_entities = set(ground_truth.lower().split())
    user_entities = set(user_text.lower().split())
    missing = truth_entities - user_entities
    missing = {w.strip(".,$") for w in missing if len(w) > 2}
    if missing:
        return f"Just to confirm, the correct info includes {', '.join(sorted(missing))}."
    return "Sounds good."


# Optional model load — only if you saved matching weights + know vocab_size.
# Mismatched checkpoint shapes → stay on fallback (expected in this course path).
weights_path = Path("correction_gpt_dpo.pt")
if not weights_path.exists():
    weights_path = Path("week2") / "correction_gpt_dpo.pt"

if weights_path.exists():
    try:
        # You must load the SAME tokenizer/vocab used when training those weights.
        # Without a saved tokenizer artifact, skip neural path for now.
        raise FileNotFoundError(
            "Saved BPE tokenizer artifact not wired yet — using rule-based fallback. "
            f"(Found weights at {weights_path}, but vocab must match training.)"
        )
    except Exception as e:
        print(f"Model not available ({e}). Using rule-based correction.")
        USE_MODEL = False
else:
    print("No correction_gpt_dpo.pt found. Using rule-based correction.")

print(f"generate_correction ready | USE_MODEL={USE_MODEL}")
print(
    "Demo:",
    generate_correction(
        "The Professional plan costs $299 per month.",
        "the professional plan is three hundred a month",
    ),
)


Model not available (Saved BPE tokenizer artifact not wired yet — using rule-based fallback. (Found weights at correction_gpt_dpo.pt, but vocab must match training.)). Using rule-based correction.
generate_correction ready | USE_MODEL=False
Demo: Just to confirm, the correct info includes 299, costs, month, per.


In [7]:
# =============================================================================
# MAIN LOOP — mic → VAD → ASR → correction → TTS
# =============================================================================
# One cycle when you finish speaking:
#   chunk from queue → VAD may emit utterance → Whisper text
#                   → generate_correction → print + speak
#

def main():
    audio_queue = queue.Queue()
    vad = VAD()
    asr = ASR(model_size="tiny.en")
    tts = TTS()

    # Mic capture on a side thread (audio_stream blocks on read)
    audio_thread = threading.Thread(target=audio_stream, args=(audio_queue,), daemon=True)
    audio_thread.start()

    # Product fact the coach should protect (later: CRM / MCP live lookup)
    ground_truth = "The Professional plan costs $299 per month."

    print("Assistant ready. Speak into the microphone... (Ctrl+C to stop)")
    try:
        while True:
            chunk = audio_queue.get()
            timestamp = time.time()
            utterance = vad.process(chunk, timestamp)
            if utterance:
                print("User spoke, transcribing...")
                user_text = asr.transcribe(utterance)
                print(f"User: {user_text}")
                correction = generate_correction(ground_truth, user_text)
                print(f"Whisper (earpiece): {correction}")
                tts.speak(correction)
    except KeyboardInterrupt:
        print("Stopping...")


# In Jupyter, call main() in a cell when you want to go live.
# if __name__ == "__main__":
#     main()
print("main() defined — run main() when mic permissions are ready.")


main() defined — run main() when mic permissions are ready.


In [8]:
# =============================================================================
# LATENCY TEST (simulated) — no mic required if you have a WAV
# =============================================================================
# Measures rough ASR + correction time on a file.
# Needs: ASR instance, generate_correction, and test_call.wav
#

from pathlib import Path

wav_path = Path("test_call.wav")
if not wav_path.exists():
    wav_path = Path("week2") / "test_call.wav"

if "asr" not in globals():
    asr = ASR(model_size="tiny.en")

ground_truth = globals().get(
    "ground_truth",
    "The Professional plan costs $299 per month.",
)

if not wav_path.exists():
    print(f"No {wav_path} found — skip live file test.")
    print("Demo without WAV: fake transcript only.")
    user_text = "the professional plan is three hundred a month"
    start = time.time()
    correction = generate_correction(ground_truth, user_text)
    print(f"Correction: {correction} ({time.time() - start:.2f}s)")
    if "tts" not in globals():
        tts = TTS()
    tts.speak(correction)
else:
    with wave.open(str(wav_path), "rb") as wf:
        audio_data = wf.readframes(wf.getnframes())
    start = time.time()
    user_text = asr.transcribe(audio_data)
    print(f"ASR: {user_text} ({time.time() - start:.2f}s)")
    start = time.time()
    correction = generate_correction(ground_truth, user_text)
    print(f"Correction: {correction} ({time.time() - start:.2f}s)")
    if "tts" not in globals():
        tts = TTS()
    tts.speak(correction)


No week2/test_call.wav found — skip live file test.
Demo without WAV: fake transcript only.
Correction: Just to confirm, the correct info includes 299, costs, month, per. (0.00s)
